# Leische prototype v3 — Stage 4 v2: a stronger gated fusion (drafted, **not run**)

Implements [suggestions/GATED_FUSION_V2.md](suggestions/GATED_FUSION_V2.md) as
a separate prototype. It pulls the committed pipeline in (§1–§8b of
`leische_pipeline.ipynb`: data, frozen folds, channels, `SharedEncoder`,
`TargetAttention`, `scatter_items`, optimiser, metrics) and adds the v2
components on top. The thesis architecture stays the reported baseline; every
v2 element is a flag, so each row of the experiment plan (§6) differs from its
predecessor by one change.

**Status:** code drafted and checked only with a 4-row CPU forward pass (the
stub check in §5). No training has been run. Run it with
`uv run --extra explore python tools/run_notebook.py --notebook leische_prototype_v3.ipynb --set LEISCHE_RUN_V3=1`.

> Every number this notebook can produce is agreement with the LLM-ensemble
> labels (Fleiss κ 0.376; human-vs-ensemble κ −0.148 on 31 gold items), never
> with human judgement. The annotators themselves agree with the shipped label
> at F1 0.49–0.58 (majority vote 0.606) — that is the ceiling.

## How v3 differs from the committed model (and from the exploration), with sources

| # | committed Stage 1–5 (`ContextAwareSarcasmModel`) | prototype v3 (`GatedFusionV2`) | why | source |
|---|---|---|---|---|
| 1 | each context item encoded **alone**, mean-pooled, projected | each item encoded **with the target in front** (`<s> target </s></s> ROLE: item </s>`), pooled over the item span only | target–item incongruity is modelled at token level, which is what every positive result on conversational sarcasm shares; late fusion never beat the baseline here | Ghosh et al. 2018 [1]; Dong et al. 2020 [3]; FigLang 2020 [2]; exploration x1/x10 |
| 2 | gate input `[t ; cᵢ]` | gate input `[t ; cᵢ ; t⊙cᵢ ; |t−cᵢ| ; sᵢ]` with per-channel **self-report** scalars sᵢ (presence, counts, kNN rate, author prior, Δt) | agreement/contradiction become linearly readable; the gate can judge reliability, not just look | ESIM matching features, Chen et al. 2017; InferSent, Conneau et al. 2017 |
| 3 | three gates, normalised over active channels | a fourth **null** channel `c₀ = 0` with its own gate | the model can say "no context helps this comment" per instance (48% of rows have no thread turn) | Flamingo's initially-closed gate, Alayrac et al. 2022; GMU, Arevalo et al. 2017 |
| 4 | no gate regularisation | **channel dropout** (p = 0.2), learned gate temperature, weak entropy penalty | stops the gate collapsing onto a prior (measured: conv weight does not track availability) | ModDrop, Neverova et al. 2016; sparsemax as an option, Martins & Astudillo 2016 |
| 5 | classifier on `[t ; c_fused]` | classifier on `[t ; c_fused ; t⊙c_fused ; |t−c_fused|]` | sarcasm is a mismatch; hand the mismatch to the classifier | ESIM / InferSent |
| 6 | retrieval = 2·k exemplar **texts** through the encoder (6–20 passes) | retrieval = **prototype contrast** over the frozen space: `W[p_sarc − p_nonsarc ; p_sarc ⊙ p_nonsarc]`, zero encoder passes | every exemplar-text variant was ≤ baseline at 60% of the compute (stages A–D, exploration x2/x11) | kNN-LM, Khandelwal et al. 2020 [8]; stage D |
| 7 | one hard label | **one head per annotator** + vote-share soft label on the main head + **cue-supervised gates** (`contextual_incongruity → g_conv`, `polarity_inversion ∧ ¬incongruity → g_null`) | the votes are the only per-row reliability signal; the cue labels are free supervision the gate never had | Davani et al. 2022 [10]; Uma et al. 2021 [9]; Lukasik et al. 2020 [12]; exploration x10 |
| 8 | fp16 + GradScaler | bf16 autocast, length-bucketed batches | stability and speed (exploration) | — |

Reference numbers [n] are the verified list in
[suggestions/IMPROVEMENTS.md](suggestions/IMPROVEMENTS.md); the design-only
references are listed at the end of `GATED_FUSION_V2.md`.

## 1 · Pull the pipeline in (§1–§8b of leische_pipeline.ipynb)

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists(), "run from the repo root"
sys.path.insert(0, str(ROOT / "tools"))
from run_notebook import load_cells, select

for _c in select(load_cells(ROOT / "leische_pipeline.ipynb"), None, ["1", "2", "3", "5", "6", "7", "8", "8b"]):
    exec(compile(_c["source"], f"<pipeline cell {_c['index']}>", "exec"), globals())
RUN_V3 = os.environ.get("LEISCHE_RUN_V3", "0") == "1"    # the plan in §6 trains only when this is set
print(f"pipeline loaded: {len(df)} rows, {len(FOLDS)} folds, train-ready={GATE.passed}; RUN_V3={RUN_V3}")

## 2 · Configuration

Every v2 element is a flag. `V3Config()` with all flags off is the committed
full model (up to the bf16 loop); the rows in §6 turn them on one group at a
time.

In [ ]:
from dataclasses import dataclass, field, asdict


@dataclass
class V3Config:
    run_name: str = "v3"
    encoder_name: str = "xlm-roberta-base"
    fold: int = 0
    seed: int = 13
    label_source: str = "adjudicated"
    # channels
    use_conv: bool = True
    use_temp: bool = True
    retrieval: str = "prototype"           # "prototype" (frozen-space contrast, no encoder pass) | "none"
    retrieval_k: int = 10
    temporal_k: int = 5
    temporal_window_hours: float | None = 48.0
    # v2 elements (§2 of GATED_FUSION_V2.md)
    target_aware_items: bool = False       # 2.1 encode each item together with the target
    matching_gate: bool = False            # 2.2 [t; c; t⊙c; |t−c|; self-report] instead of [t; c]
    null_channel: bool = False             # 2.3 a "no context" option
    channel_dropout: float = 0.0           # 2.4 probability of blanking a channel in training
    gate_entropy: float = 0.0              # 2.4 β·H(g) added to the loss (lower entropy)
    learned_gate_temperature: bool = False # 2.4
    interaction_classifier: bool = False   # 2.5 [t; fused; t⊙fused; |t−fused|]
    annotator_heads: bool = False          # 2.7
    soft_labels: bool = False              # 2.7 vote-share soft target on the main head
    cue_heads: bool = False                # 2.7 four cue logits (aux)
    cue_gate_loss: float = 0.0             # 2.7 BCE(g_conv, incongruity) + BCE(g_null, inversion-only)
    aux_weight: float = 0.5
    cue_weight: float = 0.25
    # budgets
    budget_target: int = 192
    budget_item: int = 96
    budget_selftext: int = 128
    d_model: int = 256
    mlp_hidden: int = 256
    dropout: float = 0.2
    # optimisation (committed values)
    lr_encoder: float = 2e-5
    lr_heads: float = 1e-4
    batch_size: int = 16
    grad_accum: int = 2
    max_epochs: int = 10
    patience: int = 3
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0

    def to_dict(self):
        return asdict(self)


ANNOTATOR_ORDER = ("gemma3", "qwen3", "sealion")
REPORT_DIMS = {"conv": 4, "temp": 5, "ret": 5, "null": 1}   # presence flag + scalars, see §3
print("V3Config ready")

## 3 · Per-fold inputs: self-report scalars, priors, retrieval prototypes

All statistics come from **training-fold rows only** (kNN banks, author rates
with leave-one-out, standardisation), the same leakage rules as the pipeline.
Each channel's self-report vector starts with a presence flag so an absent
channel is distinguishable from a present-but-uninformative one.

In [ ]:
from collections import Counter

Y_ADJ = LABEL_MAPS["adjudicated"]
_FOLD_INPUTS = {}


def _knn_stats(fold, names, ks=(25, 100)):
    train = fold["train"]
    T = EMB.vectors[[EMB.index[t] for t in train]]
    ty = np.array([Y_ADJ[t] for t in train], dtype=float)
    tthr = np.array([_thread_id[_thread_of[t]] for t in train])
    rate = {k: np.zeros(len(names)) for k in ks}
    top5, gap = np.zeros(len(names)), np.zeros(len(names))
    for s in range(0, len(names), 1024):
        q = names[s:s + 1024]
        sims = EMB.vectors[[EMB.index[x] for x in q]] @ T.T
        qthr = np.array([_thread_id[_thread_of[x]] for x in q])
        sims[qthr[:, None] == tthr[None, :]] = -np.inf
        order = np.argsort(-sims, axis=1)[:, :max(ks)]
        for k in ks:
            rate[k][s:s + len(q)] = ty[order[:, :k]].mean(1)
        top5[s:s + len(q)] = np.take_along_axis(sims, order[:, :5], 1).mean(1)
        # similarity gap between the best sarcastic and the best non-sarcastic neighbour
        pos = np.where(ty[None, :] > 0, sims, -np.inf).max(1); neg = np.where(ty[None, :] == 0, sims, -np.inf).max(1)
        gap[s:s + len(q)] = np.where(np.isfinite(pos) & np.isfinite(neg), pos - neg, 0.0)
    return rate, top5, gap


def _author_prior(fold, names, smoothing=2.0):
    n, pos = Counter(), Counter()
    for t in fold["train"]:
        a = _rows_by_name[t]["author_hash"]; n[a] += 1; pos[a] += Y_ADJ[t]
    base = np.mean([Y_ADJ[t] for t in fold["train"]]); train_set = set(fold["train"])
    cnt, rate = np.zeros(len(names)), np.zeros(len(names))
    for i, x in enumerate(names):
        a = _rows_by_name[x]["author_hash"]; c, p = n[a], pos[a]
        if x in train_set:
            c -= 1; p -= Y_ADJ[x]
        cnt[i] = c; rate[i] = (p + smoothing * base) / (c + smoothing)
    return cnt, rate


def fold_inputs(cfg: V3Config):
    """Self-report vectors (standardised on the training fold) and retrieval prototypes per row."""
    key = (cfg.fold, cfg.retrieval_k, cfg.temporal_k, cfg.temporal_window_hours)
    if key in _FOLD_INPUTS:
        return _FOLD_INPUTS[key]
    fold = FOLDS[cfg.fold]
    names = fold["train"] + fold["val"] + fold["test"]
    train_set = set(fold["train"])
    rate, top5, gap = _knn_stats(fold, names)
    cnt, arate = _author_prior(fold, names)
    temp = build_temporal(Config(temporal_k=cfg.temporal_k, temporal_window_hours=cfg.temporal_window_hours))
    temp_unb = build_temporal(Config(temporal_k=10, temporal_window_hours=None))
    rows_ = [_rows_by_name[x] for x in names]
    conv = np.stack([[1.0 if CONV[x] else 0.0, len(CONV[x]),
                      float(any(it.role == ROLE_ANCESTOR for it in CONV[x])), float(any(it.role == ROLE_REPLY for it in CONV[x]))]
                     for x in names], 0)
    tmp = np.stack([[1.0 if temp[x] else 0.0, np.log1p(len(temp[x])), np.log1p(len(temp_unb[x])),
                     (np.mean([it.delta_hours for it in temp[x]]) / 24.0 if temp[x] else 0.0), arate[i]]
                    for i, x in enumerate(names)], 0)
    ret = np.stack([[1.0, rate[25][i], rate[100][i], top5[i], gap[i]] for i in range(len(names))], 0)
    null = np.stack([[np.log1p(len(CONV[x]) + len(temp[x]))] for x in names], 0)
    reports = {}
    for name, arr in (("conv", conv), ("temp", tmp), ("ret", ret), ("null", null)):
        tr = np.array([x in train_set for x in names])
        mu, sd = arr[tr, 1:].mean(0), arr[tr, 1:].std(0) + 1e-6      # never standardise the presence flag
        arr = arr.copy(); arr[:, 1:] = (arr[:, 1:] - mu) / sd
        reports[name] = {x: arr[i].astype(np.float32) for i, x in enumerate(names)}
    # retrieval prototypes: similarity-weighted mean of the top-k bank embeddings (frozen space)
    protos = {}
    if cfg.retrieval != "none":
        res = build_fold_retrieval(EMB, fold["train"], names, k=cfg.retrieval_k, label_source=cfg.label_source)
        for x in names:
            qv = EMB.vectors[EMB.index[x]]
            out = []
            for bank in (res[x].sarc, res[x].nonsarc):
                if bank:
                    V = EMB.vectors[[EMB.index[b] for b in bank]]
                    w = np.clip(V @ qv, 0, None) + 1e-6
                    out.append((V * w[:, None]).sum(0) / w.sum())
                else:
                    out.append(np.zeros(EMB.vectors.shape[1]))
            protos[x] = np.concatenate(out).astype(np.float32)
    _FOLD_INPUTS[key] = (reports, protos, temp)
    return _FOLD_INPUTS[key]


def annotator_votes(row):
    by = {a["model_key"]: a.get("sarcastic") for a in (row["reliability"].get("annotators") or [])}
    votes = [int(by[m]) if isinstance(by.get(m), bool) else -1 for m in ANNOTATOR_ORDER]
    have = [v for v in votes if v >= 0]
    return votes, (sum(have) / len(have) if have else float(row["labels"]["sarcastic"]))


print("fold input builders ready")

## 4 · Dataset, collator (target-aware pairs), model, loss

`V3Collator` builds, for every context item, the pair
`<s> target </s></s> ROLE: item </s>` plus an `item_span` mask, flattened
across the batch with an owner index exactly as the pipeline's `Collator`
does — so `scatter_items`, `TargetAttention` and the temporal score bias are
reused unchanged. With `target_aware_items=False` the item is encoded alone
(the committed behaviour).

In [ ]:
ROLE_TEXT = {ROLE_SUBMISSION: "POST: ", ROLE_ANCESTOR: "PARENT: ", ROLE_REPLY: "REPLY: "}


class V3Dataset(Dataset):
    def __init__(self, cfg: V3Config, fullnames, reports, protos, temporal):
        self.cfg, self.names = cfg, list(fullnames)
        self.reports, self.protos, self.temporal = reports, protos, temporal
        self.labels = LABEL_MAPS[cfg.label_source]

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        f = self.names[i]; r = _rows_by_name[f]; lab = r["labels"]
        votes, share = annotator_votes(r)
        cues = [float(lab["cues"][k]) for k in CUE_KEYS] if lab.get("cues") else [-1.0] * 4
        return {"fullname": f, "target": r["text"],
                "conv": [(ROLE_TEXT[it.role] + it.text, it.role, int(it.is_submitter)) for it in CONV.get(f, [])] if self.cfg.use_conv else [],
                "temp": [("EARLIER POST: " + it.text, it.delta_hours) for it in self.temporal.get(f, [])] if self.cfg.use_temp else [],
                "proto": self.protos.get(f, np.zeros(2 * EMB.vectors.shape[1], np.float32)),
                "report": {c: self.reports[c][f] for c in REPORT_DIMS},
                "label": int(self.labels[f]), "votes": votes, "share": share, "cues": cues}


class V3Collator:
    def __init__(self, cfg: V3Config):
        self.cfg = cfg
        self.tok = AutoTokenizer.from_pretrained(cfg.encoder_name)
        self.bos, self.sep, self.pad = self.tok.cls_token_id, self.tok.sep_token_id, self.tok.pad_token_id

    def _ids(self, text, budget):
        return self.tok(text, add_special_tokens=False, truncation=True, max_length=budget)["input_ids"]

    def _flatten(self, samples, key, tgt_ids):
        """Flatten items over the batch; each item paired with its target when target_aware_items."""
        seqs, spans, owners, extras = [], [], [], []
        for b, s in enumerate(samples):
            for item in s[key]:
                text, *extra = item
                it = self._ids(text, self.cfg.budget_selftext if (key == "conv" and extra[0] == ROLE_SUBMISSION) else self.cfg.budget_item)
                if self.cfg.target_aware_items:
                    ids = [self.bos] + tgt_ids[b] + [self.sep, self.sep] + it + [self.sep]
                    span = [0] * (len(tgt_ids[b]) + 3) + [1] * len(it) + [0]
                else:
                    ids = [self.bos] + it + [self.sep]
                    span = [0] + [1] * len(it) + [0]
                seqs.append(ids); spans.append(span); owners.append(b); extras.append(extra)
        if not seqs:
            z = torch.zeros(0, 1, dtype=torch.long)
            return {"input_ids": z, "attention_mask": z.clone(), "item_span": z.clone(),
                    "batch_idx": torch.zeros(0, dtype=torch.long), "extras": []}
        width = max(len(x) for x in seqs)
        ids = torch.full((len(seqs), width), self.pad, dtype=torch.long)
        att = torch.zeros(len(seqs), width, dtype=torch.long); spn = torch.zeros(len(seqs), width, dtype=torch.long)
        for i, (x, sp) in enumerate(zip(seqs, spans)):
            ids[i, :len(x)] = torch.tensor(x); att[i, :len(x)] = 1; spn[i, :len(sp)] = torch.tensor(sp)
        return {"input_ids": ids, "attention_mask": att, "item_span": spn,
                "batch_idx": torch.tensor(owners, dtype=torch.long), "extras": extras}

    def __call__(self, samples):
        tgt_ids = [self._ids(s["target"], self.cfg.budget_target) for s in samples]
        width = max(len(x) for x in tgt_ids) + 2
        t_ids = torch.full((len(samples), width), self.pad, dtype=torch.long); t_att = torch.zeros(len(samples), width, dtype=torch.long)
        for i, x in enumerate(tgt_ids):
            seq = [self.bos] + x + [self.sep]; t_ids[i, :len(seq)] = torch.tensor(seq); t_att[i, :len(seq)] = 1
        y = torch.tensor([s["label"] for s in samples], dtype=torch.long)
        batch = {"fullnames": [s["fullname"] for s in samples],
                 "target": {"input_ids": t_ids, "attention_mask": t_att},
                 "labels": y, "soft": 0.5 * y.float() + 0.5 * torch.tensor([s["share"] for s in samples]),
                 "votes": torch.tensor([s["votes"] for s in samples], dtype=torch.long),
                 "cues": torch.tensor([s["cues"] for s in samples]),
                 "proto": torch.tensor(np.stack([s["proto"] for s in samples])),
                 "report": {c: torch.tensor(np.stack([s["report"][c] for s in samples])) for c in REPORT_DIMS}}
        conv = self._flatten(samples, "conv", tgt_ids); ex = conv.pop("extras")
        conv["role"] = torch.tensor([e[0] for e in ex], dtype=torch.long); conv["is_submitter"] = torch.tensor([e[1] for e in ex], dtype=torch.long)
        temp = self._flatten(samples, "temp", tgt_ids); ex = temp.pop("extras")
        temp["delta_hours"] = torch.tensor([e[0] for e in ex], dtype=torch.float)
        batch["conv"], batch["temp"] = conv, temp
        return batch


class GatedFusionV2(nn.Module):
    """Stage 4 v2 — see suggestions/GATED_FUSION_V2.md. Returns the pipeline's output dict
    (logits / gates / features / target_emb) plus annot_logits and cue_logits."""

    def __init__(self, cfg: V3Config):
        super().__init__()
        self.cfg, d = cfg, cfg.d_model
        pcfg = Config(encoder_name=cfg.encoder_name, d_model=d, pooling="mean")
        self.encoder = SharedEncoder(pcfg)                      # backbone + proj, reused from the pipeline
        self.channels = [c for c, on in (("conv", cfg.use_conv), ("temp", cfg.use_temp), ("ret", cfg.retrieval != "none")) if on]
        if cfg.null_channel:
            self.channels.append("null")
        if cfg.use_conv:
            self.conv_attn = TargetAttention(d); self.role_emb = nn.Embedding(3, d); self.submitter_emb = nn.Embedding(2, d)
        if cfg.use_temp:
            self.temp_attn = TargetAttention(d)
            self.temporal_lambda_raw = nn.Parameter(torch.tensor(math.log(math.expm1(0.0289))))
        if cfg.retrieval != "none":
            self.ret_proj = nn.Sequential(nn.Linear(2 * EMB.vectors.shape[1], d), nn.GELU(), nn.Linear(d, d))
        gate_in = (4 * d if cfg.matching_gate else 2 * d)
        self.gate = nn.ModuleDict({c: nn.Linear(gate_in + (REPORT_DIMS[c] if cfg.matching_gate else 0), 1) for c in self.channels})
        self.log_temp = nn.Parameter(torch.zeros(()), requires_grad=cfg.learned_gate_temperature)
        feat_dim = d * (4 if cfg.interaction_classifier else 2) if self.channels else d
        self.classifier = nn.Sequential(nn.Dropout(cfg.dropout), nn.Linear(feat_dim, cfg.mlp_hidden), nn.GELU(),
                                        nn.Dropout(cfg.dropout), nn.Linear(cfg.mlp_hidden, 2))
        self.annot_heads = nn.ModuleList([nn.Linear(feat_dim, 2) for _ in ANNOTATOR_ORDER]) if cfg.annotator_heads else None
        self.cue_head = nn.Linear(feat_dim, 4) if cfg.cue_heads else None

    @property
    def temporal_lambda(self):
        return F.softplus(self.temporal_lambda_raw)

    def _items(self, part, batch_size):
        if part["input_ids"].shape[0] == 0:
            dev = next(self.parameters()).device
            return torch.zeros(batch_size, 1, self.cfg.d_model, device=dev), torch.zeros(batch_size, 1, dtype=torch.bool, device=dev)
        hs = self.encoder.backbone(input_ids=part["input_ids"], attention_mask=part["attention_mask"]).last_hidden_state
        span = part["item_span"].unsqueeze(-1).to(hs.dtype)          # pool the item tokens only (§2.1)
        pooled = (hs * span).sum(1) / span.sum(1).clamp(min=1e-6)
        return scatter_items(self.encoder.proj(pooled), part["batch_idx"], batch_size)

    def forward(self, batch):
        cfg = self.cfg
        t = self.encoder(batch["target"]["input_ids"], batch["target"]["attention_mask"])
        B = t.shape[0]
        chans, reports = {}, {c: batch["report"][c].to(t.dtype) for c in self.channels}
        if cfg.use_conv:
            items, mask = self._items(batch["conv"], B)
            if batch["conv"]["input_ids"].shape[0] > 0:
                extra, _ = scatter_items(self.role_emb(batch["conv"]["role"]) + self.submitter_emb(batch["conv"]["is_submitter"]),
                                         batch["conv"]["batch_idx"], B)
                items = items + extra
            chans["conv"] = self.conv_attn(t, items, mask)
        if cfg.use_temp:
            items, mask = self._items(batch["temp"], B)
            bias = None
            if batch["temp"]["input_ids"].shape[0] > 0:
                lam_dt, _ = scatter_items((-self.temporal_lambda * batch["temp"]["delta_hours"]).unsqueeze(-1), batch["temp"]["batch_idx"], B)
                bias = lam_dt.squeeze(-1).to(items.dtype)
            chans["temp"] = self.temp_attn(t, items, mask, score_bias=bias)
        if cfg.retrieval != "none":
            p = batch["proto"].to(t.dtype); h = p.shape[-1] // 2
            chans["ret"] = self.ret_proj(torch.cat([p[:, :h] - p[:, h:], p[:, :h] * p[:, h:]], -1))
            chans["ret"] = torch.where((reports["ret"][:, :1] > 0), chans["ret"], torch.zeros_like(chans["ret"]))
        if cfg.null_channel:
            chans["null"] = torch.zeros_like(t)
        if self.training and cfg.channel_dropout > 0:                    # §2.4 channel dropout
            for c in ("conv", "temp", "ret"):
                if c in chans:
                    drop = torch.rand(B, 1, device=t.device) < cfg.channel_dropout
                    chans[c] = torch.where(drop, torch.zeros_like(chans[c]), chans[c])
                    reports[c] = torch.where(drop, torch.zeros_like(reports[c]), reports[c])   # presence flag → 0
        out = {"target_emb": t}
        if not self.channels:
            feats, out["gates"] = t, None
        else:
            logits = []
            for c in self.channels:
                v = chans[c]
                g_in = torch.cat([t, v, t * v, (t - v).abs(), reports[c]], -1) if cfg.matching_gate else torch.cat([t, v], -1)
                logits.append(self.gate[c](g_in))
            raw = torch.sigmoid(torch.cat(logits, -1) / self.log_temp.exp())
            g = raw / raw.sum(-1, keepdim=True).clamp(min=1e-6)
            fused = sum(g[:, i:i + 1] * chans[c] for i, c in enumerate(self.channels))
            feats = torch.cat([t, fused, t * fused, (t - fused).abs()], -1) if cfg.interaction_classifier else torch.cat([t, fused], -1)
            out["gates"] = g
            for c, v in chans.items():
                out[f"c_{c}"] = v
        out["features"], out["logits"] = feats, self.classifier(feats)
        if self.annot_heads is not None:
            out["annot_logits"] = torch.stack([h(feats) for h in self.annot_heads], 1)
        if self.cue_head is not None:
            out["cue_logits"] = self.cue_head(feats)
        return out


def v3_loss(out, batch, cfg: V3Config, class_w, model):
    logits, y = out["logits"], batch["labels"]
    if cfg.soft_labels:
        tgt = batch["soft"]
        per = -(torch.stack([1 - tgt, tgt], -1) * F.log_softmax(logits, -1)).sum(-1) * class_w[y]
    else:
        per = F.cross_entropy(logits, y, weight=class_w, reduction="none")
    loss = per.mean()
    if cfg.annotator_heads:
        v = batch["votes"]; m = v >= 0
        if m.any():
            loss = loss + cfg.aux_weight * F.cross_entropy(out["annot_logits"][m], v[m], weight=class_w)
    cues = batch["cues"]; cm = cues[:, 0] >= 0
    if cfg.cue_heads and cm.any():
        loss = loss + cfg.cue_weight * F.binary_cross_entropy_with_logits(out["cue_logits"][cm], cues[cm])
    g = out["gates"]
    if g is not None and cfg.cue_gate_loss > 0 and cm.any():
        ch = model.channels
        eps = 1e-4
        if "conv" in ch:
            gc = g[cm, ch.index("conv")].float().clamp(eps, 1 - eps)
            loss = loss + cfg.cue_gate_loss * F.binary_cross_entropy(gc, cues[cm, CUE_KEYS.index("contextual_incongruity")])
        if "null" in ch:
            inv_only = (cues[cm, CUE_KEYS.index("polarity_inversion")] > 0) & (cues[cm, CUE_KEYS.index("contextual_incongruity")] == 0)
            gn = g[cm, ch.index("null")].float().clamp(eps, 1 - eps)
            loss = loss + cfg.cue_gate_loss * F.binary_cross_entropy(gn, inv_only.float())
    if g is not None and cfg.gate_entropy > 0:
        loss = loss + cfg.gate_entropy * (-(g.float().clamp(min=1e-6) * g.float().clamp(min=1e-6).log()).sum(-1)).mean()
    return loss


print("dataset / collator / model / loss ready")

## 5 · Training loop, run helper, and a CPU stub check

The loop mirrors the exploration notebook (bf16 autocast, early stopping on
validation F1@0.5, best weights restored, validation-chosen threshold and
temperature reported beside the 0.5 decision). `run_v3` is skip-if-done with
default-tolerant config comparison. The stub check below runs one 4-row forward
pass on the CPU with the real encoder — it verifies shapes and masks, nothing
about accuracy.

In [ ]:
def make_v3_loaders(cfg: V3Config):
    fold = FOLDS[cfg.fold]
    reports, protos, temporal = fold_inputs(cfg)
    collate = V3Collator(cfg)
    loaders = {}
    for part in ("train", "val", "test"):
        ds = V3Dataset(cfg, fold[part], reports, protos, temporal)
        loaders[part] = DataLoader(ds, batch_size=cfg.batch_size if part == "train" else cfg.batch_size * 2,
                                   shuffle=(part == "train"), generator=torch.Generator().manual_seed(cfg.seed),
                                   collate_fn=collate)
    return loaders


def predict_v3(model, loader):
    model.eval(); rows = []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                out = model(batch)
            lg = out["logits"].float(); margin = (lg[:, 1] - lg[:, 0]).cpu().numpy(); prob = 1 / (1 + np.exp(-margin))
            gates = out["gates"].float().cpu().numpy() if out["gates"] is not None else None
            for i, f in enumerate(batch["fullnames"]):
                r = {"reddit_fullname": f, "y_true": int(batch["labels"][i]), "prob": float(prob[i]), "margin": float(margin[i]), "pred": int(prob[i] >= 0.5)}
                if gates is not None:
                    for j, c in enumerate(model.channels):
                        r[f"gate_{c}"] = float(gates[i, j])
                rows.append(r)
    return pd.DataFrame(rows)


def train_v3(model, loaders, cfg: V3Config, class_w, log=True):
    model.to(DEVICE)
    opt = build_optimizer(model, Config(lr_encoder=cfg.lr_encoder, lr_heads=cfg.lr_heads))
    sched = build_scheduler(opt, max(1, len(loaders["train"]) * cfg.max_epochs // cfg.grad_accum), cfg.warmup_ratio)
    w = torch.tensor(class_w, dtype=torch.float, device=DEVICE)
    best, best_state, left = -1.0, None, cfg.patience
    hist = {"train_loss": [], "val_f1": [], "epoch_sec": []}
    for epoch in range(cfg.max_epochs):
        t0 = time.time(); model.train(); losses = []; opt.zero_grad(set_to_none=True)
        for step, batch in enumerate(loaders["train"]):
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = v3_loss(model(batch), batch, cfg, w, model) / cfg.grad_accum
            loss.backward()
            if (step + 1) % cfg.grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip); opt.step(); opt.zero_grad(set_to_none=True); sched.step()
            losses.append(float(loss.item()) * cfg.grad_accum)
        val = predict_v3(model, loaders["val"]); f1 = safe_f1(val["y_true"].to_numpy(), val["pred"].to_numpy())
        hist["train_loss"].append(float(np.mean(losses))); hist["val_f1"].append(f1); hist["epoch_sec"].append(round(time.time() - t0, 1))
        if log:
            print(f"    epoch {epoch}: loss {hist['train_loss'][-1]:.4f} val_f1@0.5 {f1:.4f} ({hist['epoch_sec'][-1]:.0f}s)", flush=True)
        if f1 > best:
            best, left, best_state = f1, cfg.patience, copy.deepcopy(model.state_dict())
        else:
            left -= 1
            if left <= 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    hist.update(best_val_f1=best, best_epoch=int(np.argmax(hist["val_f1"])) + 1, epochs_run=len(hist["val_f1"]))
    return hist


V3_DIR = RESULTS / "prototype_v3"
_V3_KEYS = [k for k in V3Config.__dataclass_fields__ if k != "run_name"]


def run_v3(cfg: V3Config, verbose=True):
    out_dir = V3_DIR / f"{cfg.run_name}-f{cfg.fold}-s{cfg.seed}"
    rj = out_dir / "run.json"
    if rj.exists():
        saved, dflt = json.loads(rj.read_text(encoding="utf-8"))["config"], V3Config().to_dict()
        if {k: saved.get(k, dflt[k]) for k in _V3_KEYS} == {k: cfg.to_dict()[k] for k in _V3_KEYS}:
            print(f"{cfg.run_name} f{cfg.fold} s{cfg.seed}: loaded"); return json.loads(rj.read_text(encoding="utf-8"))["summary"]
    t0 = time.time(); set_seed(cfg.seed); torch.cuda.reset_peak_memory_stats()
    loaders = make_v3_loaders(cfg)
    model = GatedFusionV2(cfg)
    hist = train_v3(model, loaders, cfg, class_weights(FOLDS[cfg.fold]["train"], LABEL_MAPS[cfg.label_source]), log=verbose)
    val = predict_v3(model, loaders["val"])
    thr, _ = best_threshold(val["y_true"].to_numpy(), val["prob"].to_numpy())
    temp = fit_temperature(val["margin"].to_numpy(), val["y_true"].to_numpy())
    pred = predict_v3(model, loaders["test"])
    pred["pred_tuned"] = (pred["prob"] >= thr).astype(int); pred["prob_cal"] = 1 / (1 + np.exp(-pred["margin"] / temp))
    pred = pred.merge(META, on="reddit_fullname", how="left")
    y = pred["y_true"].to_numpy(); m = safe_metrics(y, pred["pred"].to_numpy()); mt = safe_metrics(y, pred["pred_tuned"].to_numpy())
    gcols = [c for c in pred.columns if c.startswith("gate_")]
    summary = {"run": cfg.run_name, "fold": cfg.fold, "seed": cfg.seed, "f1": m["f1"], "precision": m["precision"], "recall": m["recall"],
               "f1_tuned": mt["f1"], "threshold": thr, **rank_metrics(y, pred["prob"].to_numpy()), "ece": ece(y, pred["prob"].to_numpy()),
               "best_val_f1": hist["best_val_f1"], "best_epoch": hist["best_epoch"], "epochs_run": hist["epochs_run"],
               **{f"{c}_mean": float(pred[c].mean()) for c in gcols}, **{f"{c}_std": float(pred[c].std()) for c in gcols},
               "min": round((time.time() - t0) / 60, 1), "peak_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 1)}
    out_dir.mkdir(parents=True, exist_ok=True)
    pred.to_csv(out_dir / "predictions.csv", index=False)
    rj.write_text(json.dumps({"config": cfg.to_dict(), "summary": summary, "history": hist, "dataset_identity": IDENTITY,
                              "label_authority": LABEL_AUTHORITY}, indent=2), encoding="utf-8")
    if verbose:
        print(f"{cfg.run_name} f{cfg.fold} s{cfg.seed}: F1@0.5 {m['f1']:.4f} F1@val-thr {mt['f1']:.4f} AUPRC {summary['auprc']:.4f} "
              f"AUROC {summary['auroc']:.4f} ECE {summary['ece']:.3f} | gates " + " ".join(f"{c[5:]}={pred[c].mean():.2f}±{pred[c].std():.2f}" for c in gcols)
              + f" | {summary['min']} min, {summary['peak_gb']} GB", flush=True)
    del model, loaders; torch.cuda.empty_cache()
    return summary


def stub_check(cfg: V3Config, n=4):
    """One forward pass on n training rows, on the CPU, with the real encoder: shapes and masks only."""
    fold = FOLDS[cfg.fold]; reports, protos, temporal = fold_inputs(cfg)
    ds = V3Dataset(cfg, fold["train"][:n], reports, protos, temporal)
    batch = V3Collator(cfg)([ds[i] for i in range(n)])
    model = GatedFusionV2(cfg).eval()
    with torch.no_grad():
        out = model(batch)
    g = out["gates"]
    print(f"{cfg.run_name}: logits {tuple(out['logits'].shape)}, features {tuple(out['features'].shape)}, "
          f"gates {None if g is None else tuple(g.shape)} sum→{None if g is None else g.sum(-1).tolist()} channels={model.channels} "
          f"conv items {batch['conv']['input_ids'].shape} temp items {batch['temp']['input_ids'].shape}")
    loss = v3_loss(out, batch, cfg, torch.tensor([1.0, 1.0]), model)
    assert torch.isfinite(loss), "non-finite loss"
    print(f"  loss {float(loss):.4f} (finite)")
    return out


print("loop + run helper ready")

## 6 · The experiment plan (`GATED_FUSION_V2.md` §5) — trains only with `LEISCHE_RUN_V3=1`

Each row differs from the previous one by one group of changes. Fold 0 / seed 13
first; the seed and 5-fold cells below run only for the best row.

In [ ]:
V3_ROWS = [
    V3Config(run_name="v2-0-committed-bf16"),                                             # the committed model in this loop
    V3Config(run_name="v2-a-gate", matching_gate=True, null_channel=True, channel_dropout=0.2,
             learned_gate_temperature=True, gate_entropy=0.01, interaction_classifier=True),
    V3Config(run_name="v2-b-target-aware", matching_gate=True, null_channel=True, channel_dropout=0.2,
             learned_gate_temperature=True, gate_entropy=0.01, interaction_classifier=True, target_aware_items=True),
    V3Config(run_name="v2-c-votes", matching_gate=True, null_channel=True, channel_dropout=0.2,
             learned_gate_temperature=True, gate_entropy=0.01, interaction_classifier=True, target_aware_items=True,
             annotator_heads=True, soft_labels=True, cue_heads=True, cue_gate_loss=0.1),
    V3Config(run_name="v2-d-large", encoder_name="xlm-roberta-large", batch_size=8, grad_accum=4, lr_encoder=1e-5,
             matching_gate=True, null_channel=True, channel_dropout=0.2, learned_gate_temperature=True, gate_entropy=0.01,
             interaction_classifier=True, target_aware_items=True, annotator_heads=True, soft_labels=True, cue_heads=True, cue_gate_loss=0.1),
]

# the stub check is cheap and safe to run any time (CPU, 4 rows): it is how you know the code is not broken
if os.environ.get("LEISCHE_STUB_CHECK", "0") == "1":
    _dev = DEVICE
    DEVICE = torch.device("cpu")
    for _cfg in V3_ROWS[:4]:
        stub_check(V3Config(**{**_cfg.to_dict(), "encoder_name": "xlm-roberta-base"}))
    DEVICE = _dev

rows = []
if RUN_V3:
    for cfg in V3_ROWS:
        try:
            rows.append(run_v3(cfg))
        except torch.OutOfMemoryError:
            print(f"{cfg.run_name}: OOM under the VRAM cap — skipped"); torch.cuda.empty_cache()
    grid = pd.DataFrame(rows)
    print(grid.round(4).to_string(index=False))
    grid.to_csv(V3_DIR / "grid-fold0-seed13.csv", index=False)
else:
    print("RUN_V3 is off — nothing trained. Set LEISCHE_RUN_V3=1 to execute the plan.")

## 7 · Seeds and five folds for the best row (only after §6 has run)

Pick by **validation** F1, never by the test fold; compare against
`v2-0-committed-bf16` at the same seeds and folds, with the pipeline's paired
bootstrap. Then plot the null gate against the number of context turns — the
figure the per-instance-gating claim needs.

In [ ]:
if RUN_V3 and rows:
    grid = pd.DataFrame(rows)
    base_rows = grid[grid["run"] != "v2-0-committed-bf16"]
    BEST = base_rows.sort_values("best_val_f1", ascending=False)["run"].iloc[0]
    best_cfg = next(c for c in V3_ROWS if c.run_name == BEST)
    print("best on validation:", BEST)
    seed_rows = list(rows)
    for s in (42, 7):
        for c in (best_cfg, V3_ROWS[0]):
            seed_rows.append(run_v3(V3Config(**{**c.to_dict(), "seed": s})))
    cv_rows = []
    for fi in range(len(FOLDS)):
        for c in (best_cfg, V3_ROWS[0]):
            cv_rows.append(run_v3(V3Config(**{**c.to_dict(), "fold": fi})))
    cv = pd.DataFrame(cv_rows)
    print(cv.groupby("run").agg(f1=("f1", "mean"), f1_std=("f1", "std"), auprc=("auprc", "mean"), auroc=("auroc", "mean"), ece=("ece", "mean")).round(4))
    # null gate vs context availability (the per-instance claim)
    p = pd.concat([pd.read_csv(V3_DIR / f"{BEST}-f{fi}-s13" / "predictions.csv") for fi in range(len(FOLDS))])
    if "gate_null" in p:
        p["n_turns"] = p["reddit_fullname"].map(lambda f: len(CONV.get(f, [])) - (1 if _rows_by_name[f]["context"].get("submission") else 0))
        print(p.groupby(pd.cut(p["n_turns"], [-1, 0, 1, 100], labels=["post only", "1 turn", "2+ turns"]), observed=True)
              [[c for c in p.columns if c.startswith("gate_")]].mean().round(3))
else:
    print("nothing to confirm yet")

## 8 · What would count as success, and what would not

- **Success:** F1 ≥ the exploration's 0.405 on 5 folds with the gate's null
  mass rising monotonically as context turns fall (post-only rows highest),
  and the conversational gate opening on rows the annotators flagged as
  `contextual_incongruity`. That would make the thesis's per-instance gating a
  supported mechanism with a figure behind it.
- **Partial success:** F1 ≥ baseline but flat gates — the votes and the
  target-aware encoding carry the gain and the gate is decoration; report
  early fusion as the model and the gate as an ablation.
- **Failure:** F1 at baseline — the channel structure itself is the limit on
  these labels.
- **Never a success:** any F1 above ~0.5 against the adjudicated label. The
  annotators' majority vote scores 0.606; check for leakage before believing
  it.